In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_GAMES = "grandmasters.silver.game_events"
SILVER_MOVES = "grandmasters.silver.game_move_events"
SILVER_PLAYER_METRICS = "grandmasters.silver.player_metrics"
SILVER_PLAYER_MOVES = "grandmasters.silver.player_moves"

DEVELOPMENT_MODE = True

In [0]:
# ============================================================
# READ SILVER TABLES
#
# Purpose:
# Read validated Silver tables to build player-level metrics.
# ============================================================

games_df = spark.table(SILVER_GAMES)

moves_df = spark.table(SILVER_MOVES)

display(games_df.limit(10))

display(moves_df.limit(10))

In [0]:
# ============================================================
# PLAYER MOVE METRICS
#
# Purpose:
# Calculate move-level metrics such as think time and clock loss.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

move_window = (
    Window
    .partitionBy("game_id")
    .orderBy("move_seq")
)

player_moves_df = (

    moves_df

    .withColumn(
        "previous_white_clock",
        F.lag("white_clock_secs").over(move_window)
    )

    .withColumn(
        "previous_black_clock",
        F.lag("black_clock_secs").over(move_window)
    )

    .withColumn(
        "player",
        F.when(
            F.split(F.col("fen"), " ").getItem(1) == "w",
            "Black"   # if it's white's turn NOW, black just moved
        ).otherwise("White")  # if it's black's turn now, white just moved
    )

    .withColumn(
        "remaining_clock_secs",
        F.when(
            F.col("player") == "White",
            F.col("white_clock_secs")
        ).otherwise(
            F.col("black_clock_secs")
        )
    )

    .withColumn(
        "think_time_secs",
        F.when(
            F.col("player") == "White",
            F.col("previous_white_clock") - F.col("white_clock_secs")
        ).otherwise(
            F.col("previous_black_clock") - F.col("black_clock_secs")
        )
    )

    .withColumn(
        "clock_loss_secs",
        F.col("think_time_secs")
    )

)

display(player_moves_df)

In [0]:
# ============================================================
# WRITE SILVER PLAYER MOVES
# ============================================================

(
    player_moves_df
    .select(
        "game_id",
        "move_seq",
        "player",
        "think_time_secs",
        "clock_loss_secs",
        "remaining_clock_secs",
        "ingested_at"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_PLAYER_MOVES)
)

print(f"Successfully wrote {player_moves_df.count()} rows to {SILVER_PLAYER_MOVES}")